In [ ]:
#### VCPI-dataset
# !pip install seaborn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
# !pip install loguru
# 1. Point your path directly to the 'src' directory inside the repo
repo_src_path = os.path.abspath("../BioXAI_Hackathon/vcpi-prediction-contest-2026/src")

# 2. Add the src directory to Python's search path
if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

# 3. Now, import straight from the package name (dropping the 'src.' prefix!)
from vcpi_prediction_contest.expression import counts_to_expression

# Set up clean plotting styles for Jupyter
%matplotlib inline
sns.set_theme(style="whitegrid")

# Load the file (replace with your exact local path if necessary)
df_qnu_raw = pd.read_parquet("../BioXAI_Hackathon/VCPI_data/vcpi_tvc-qnu-012_counts.parquet") #### VCPI_metadata
df_bhr_raw = pd.read_parquet("../BioXAI_Hackathon/VCPI_data/vcpi_tvc-bhr-009_counts.parquet")
df_kdl_raw = pd.read_parquet("../BioXAI_Hackathon/VCPI_data/vcpi_tvc-kdl-010_counts.parquet")
# 1. Combine your raw counts horizontally (samples as columns, gene_id as index)
# DO NOT normalize or log-transform them yourself yet; pass the raw UMI counts!
raw_counts_matrix = pd.concat([df_bhr_raw, df_kdl_raw, df_qnu_raw], axis=1, join='inner')
raw_counts_matrix.to_csv('../BioXAI_Hackathon/code_generated_outputs/raw_count_matrix_all3.csv', index = False)
# 2. Prepare the metadata subset containing your filtered 10uM conditions
# Make sure 'sequenced_id' matches the column names in raw_counts_matrix
# meta_subset = master_df[['sequenced_id', 'compound']].drop_duplicates()


# # Display the first few rows to confirm loading
# meta_subset.head()

In [ ]:
raw_counts_matrix['gene_id']

In [ ]:
raw_counts_matrix.shape

In [ ]:
count_matrix = df.set_index('gene_id')

# 2. Transpose so rows = samples (sequenced_id), columns = genes
counts_df = count_matrix.T
counts_df.index.name = 'sequenced_id'

# 3. Convert the index to integer if your metadata 'sequenced_id' is numeric
counts_df.index = counts_df.index.astype(int)

In [ ]:
counts_df

In [ ]:
############ Quick QC 
# 1. Filter out lowly expressed genes (optional but highly recommended to speed up ML)
# Keep genes that have at least 10 counts in at least 5% of the samples
min_reads = 10
min_samples = int(0.05 * len(counts_df))
genes_to_keep = (counts_df >= min_reads).sum(axis=0) >= min_samples
counts_filtered = counts_df.loc[:, genes_to_keep]

# 2. Log-normalize using Counts Per Million (CPM) adjustment
# (Dividing by row totals and taking log2 stabilizes variance for trees/neural nets)
row_sums = counts_filtered.sum(axis=1)
counts_normalized = counts_filtered.div(row_sums, axis=0) * 1e6
counts_normalized = np.log2(counts_normalized + 1)

In [ ]:
counts_normalized

In [5]:
###### open one compund file 
compounds_qnu = pd.read_csv("../BioXAI_Hackathon/VCPI_data/compounds-tvc-qnu-012-2026-05-29.csv") #### VCPI_compound
compounds_bhr = pd.read_csv("../BioXAI_Hackathon/VCPI_data/compounds-tvc-bhr-009-2026-05-29.csv")
compounds_kdl = pd.read_csv("../BioXAI_Hackathon/VCPI_data/compounds-tvc-kdl-010-2026-05-29.csv")

compounds_df = pd.concat([compounds_bhr, compounds_qnu, compounds_kdl], axis=0, ignore_index=True).drop_duplicates(subset=['compound'])
compounds_df.to_csv('../BioXAI_Hackathon/code_generated_outputs/compounds_all.csv', index = False)
print("Number of unique compounds are: ", len(compounds_df.user_compound_id.unique()))
compounds_df.head(2)

Number of unique compounds are:  14031


,compound,user_compound_id,smiles,purity_pct,molecular_weight,log_p,tpsa,inchi_key,num_rotatable_bonds,num_h_acceptors,num_h_donors,num_atoms,num_bonds
0,3c298cd3-0ba2-4e86-9d18-c03a08d47314,9258062,Cc1nc(N2CCN(S(=O)(=O)CCOc3ccccc3)CC2)[nH]c(=O)c1F,NaN,396.44,0.75,95.60,DCBFYUZEIWVCJE-UHFFFAOYSA-N,6,6,1,27,29
1,801ef874-2b7e-424e-b8b8-ece89f093b0c,9255050,O=C(CCCS(=O)(=O)c1ccccc1)N1CCN(c2ccccc2)CC1,NaN,372.49,2.59,57.69,QQFQAYZDKYDRDF-UHFFFAOYSA-N,6,4,0,26,28


In [ ]:
compounds_df.columns

In [3]:
#### Combine metadata files
import pandas as pd

# 1. Load the three metadata files
metadata_qnu = pd.read_csv("../BioXAI_Hackathon/VCPI_data/metadata-tvc-qnu-012.csv")
metadata_bhr = pd.read_csv("../BioXAI_Hackathon/VCPI_data/metadata-tvc-bhr-009.csv")
metadata_kdl = pd.read_csv("../BioXAI_Hackathon/VCPI_data/metadata-tvc-kdl-010.csv")

# 2. Combine them vertically into a single master metadata DataFrame
# ignore_index=True resets the row index from 0 to the new total length
metadata_df = pd.concat([metadata_qnu, metadata_bhr, metadata_kdl], axis=0, ignore_index=True)

# 3. Clean string white-spaces to ensure matching accuracy
metadata_df['user_compound_id'] = metadata_df['user_compound_id'].astype(str).str.strip()

# =====================================================================
# QC & CRITERIA FILTERING (From our previous distribution analyses)
# =====================================================================

# Criterion A: Mitochondrial expression QC filter (keep healthy cells <= 10%)
meta_qc = metadata_df[metadata_df['percent_mitochondrial'] <= 10.0]

# Criterion B: Isolate 10,000 nM active treatments OR DMSO vehicle controls
valid_samples_mask = (
    ((meta_qc['compound_concentration'] == 10000) & (meta_qc['is_control'] == False) & (meta_qc['user_compound_id'].str.upper() != 'DMSO')) |
    ((meta_qc['is_control'] == True) | (meta_qc['user_compound_id'].str.upper() == 'DMSO'))
)

# 4. Create your filtered sample-level master data frame
master_df = meta_qc[valid_samples_mask].copy()

# 5. Convert sequenced_id to string to match standard raw count column headers
master_df['sequenced_id'] = master_df['sequenced_id'].astype(str)
# =====================================================================
# EXTRACT THE METADATA SUBSET FOR counts_to_expression()
# =====================================================================
# This exact mapping can now be safely passed directly to the contest function
meta_subset = master_df[['sequenced_id', 'compound']].drop_duplicates()

print(f"Total samples loaded from all 3 files: {len(metadata_df)}")
print(f"Total samples remaining after filtering: {len(master_df)}")
print(f"Shape of the final meta_subset mapping: {meta_subset.shape}")
metadata_df.head(2)

Total samples loaded from all 3 files: 70200
Total samples remaining after filtering: 32384
Shape of the final meta_subset mapping: (32384, 2)


,sequenced_id,job_id,container_id,column_id,row_id,is_edge,compound,user_compound_id,compound_concentration,compound_concentration_unit,...,total_sequenced_reads,total_umi_count,ngenes3,n_mapped,percent_mapped,percent_rrna_removed,percent_mitochondrial,unassigned_multimapping,unassigned_nofeatures,percent_duplicated
0,116471544,tvc-qnu-012,1839025,1,1,True,e24cec84-a4a0-42b4-8914-67d02876a890,DMSO,0.15,%,...,2301152,888404,10858,1770243,86.07,10.63,5.215195,349502,109315,0.649494
1,116471545,tvc-qnu-012,1839025,1,2,True,9815b4fc-f243-497b-b5a1-408d0ac26e60,1306737,10000.00,nM,...,3087310,1222144,11371,2442721,86.80,8.85,4.305712,464527,122958,0.635829


In [ ]:
metadata_df.columns

In [ ]:
metadata_df.head(2)

In [ ]:
meta_qc.head(2)

In [ ]:
long_expression_df = counts_to_expression(raw_counts_matrix, meta_subset)

print("Long expression table head:")
print(long_expression_df.head())

In [6]:
# 1. Merge metadata with compound features
# We select the columns we need from compounds to prevent column clashing
compounds_subset = compounds_df[['compound', 'smiles', 'molecular_weight', 'log_p', 
                                 'tpsa', 'num_rotatable_bonds', 'num_h_acceptors', 
                                 'num_h_donors', 'num_atoms', 'num_bonds']]

meta_compounds = pd.merge(metadata_df, compounds_subset, on='compound', how='inner')

# 2. Merge the resulting df with your normalized counts matrix
master_df = pd.merge(meta_compounds, counts_normalized, on='sequenced_id', how='inner')
master_df.to_csv('../BioXAI_Hackathon/code_generated_outputs/master_df_all.csv', index = False)
print(f"Master DataFrame shape: {master_df.shape}")

NameError: name 'counts_normalized' is not defined

In [ ]:
master_df[['sequenced_id', 'compound', 'total_umi_count']]

In [ ]:
import pandas as pd

# 1. Standardize text strings to lower-case to avoid capitalization issues
master_df['user_compound_id'] = master_df['user_compound_id'].astype(str).str.strip()

# 2. Filter for active compounds at exactly 10,000 nM (10 uM)
active_10uM_mask = (
    (master_df['compound_concentration'] == 10000) & 
    (master_df['is_control'] == False) & 
    (master_df['user_compound_id'].str.upper() != 'DMSO')
)
df_active_10uM = master_df[active_10uM_mask].copy()

# 3. Filter for DMSO / Vehicle Controls
control_mask = (master_df['is_control'] == True) | (master_df['user_compound_id'].str.upper() == 'DMSO')
df_controls = master_df[control_mask].copy()

print(f"Total samples in dataset: {len(master_df)}")
print(f"Active treatment samples at 10,000 nM: {len(df_active_10uM)}")
print(f"Control (DMSO) samples: {len(df_controls)}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Extract only the gene columns from your master_df 
# (Excluding the metadata/chemical property columns you listed earlier)
metadata_cols = [
    'sequenced_id', 'job_id', 'container_id', 'column_id', 'row_id', 'is_edge', 
    'compound', 'user_compound_id', 'compound_concentration', 'compound_concentration_unit', 
    'cell_line', 'timepoint', 'condition', 'percent_volume_dmso', 'is_control', 
    'seeded_cell_count', 'total_sequenced_reads', 'total_umi_count', 'ngenes3', 
    'n_mapped', 'percent_mapped', 'percent_rrna_removed', 'percent_mitochondrial', 
    'unassigned_multimapping', 'unassigned_nofeatures', 'percent_duplicated', 
    'smiles', 'molecular_weight', 'log_p', 'tpsa', 'num_rotatable_bonds', 
    'num_h_acceptors', 'num_h_donors', 'num_atoms', 'num_bonds'
]

# Find all columns that are NOT metadata (these are your genes)
gene_cols = [col for col in master_df.columns if col not in metadata_cols]

X_genes = master_df[gene_cols]

# 2. Standardize the gene expression data (mean=0, variance=1)
X_scaled = StandardScaler().fit_transform(X_genes)

# 3. Compute PCA down to 2 dimensions
pca = PCA(n_components=2)
pca_results = pca.fit_transform(X_scaled)

# 4. Create a plotting DataFrame
df_pca = pd.DataFrame(pca_results, columns=['PC1', 'PC2'])
df_pca['Concentration (nM)'] = master_df['compound_concentration'].fillna(0).values
df_pca['Is_Control'] = master_df['is_control'].values

# Label DMSO explicitly for clarity in the plot
df_pca.loc[df_pca['Is_Control'] == True, 'Concentration (nM)'] = 'DMSO Control'

# 5. Plot the full data space
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x='PC1', y='PC2', 
    hue='Concentration (nM)', 
    data=df_pca, 
    palette='viridis', 
    alpha=0.7, 
    edgecolor=None
)

plt.title('PCA of Whole Screen Transcriptomic Profiles by Compound Concentration', fontsize=14)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance Explained)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance Explained)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Dosage Condition')
plt.tight_layout()
plt.show()

In [ ]:
####### Plot mitochondrial expression
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing mitochondrial concentration to establish a QC threshold
plt.figure(figsize=(8, 5))
sns.histplot(master_df['percent_mitochondrial'], bins=50, kde=True, color='crimson')
plt.axvline(x=10, color='black', linestyle='--', label='Suggested Upper Threshold (10%)')
plt.title('Distribution of Percent Mitochondrial Reads Across Samples')
plt.xlabel('Percent Mitochondrial (%)')
plt.ylabel('Count')
plt.legend()
plt.show()

In [1]:
######## Generate features based on SMILES #####
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# 1. Initialize the Morgan Generator once (outside your functions to save memory)
# radius=2 corresponds to the classic ECFP4 fingerprint
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
master_df = pd.read_csv('../BioXAI_Hackathon/code_generated_outputs/master_df_all.csv')
def smiles_to_fingerprint_clean(smiles):
    try:
        # Check if the SMILES string is valid/readable
        if pd.isna(smiles) or not isinstance(smiles, str):
            return [0] * 1024
            
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            # 2. Generate the fingerprint using the new object syntax
            fp = morgan_gen.GetFingerprint(mol)
            return list(fp)
    except Exception as e:
        pass
    
    # Return a vector of zeros if structural parsing fails
    return [0] * 1024

# Extract unique smiles to save compute time, map them, and build an array
unique_smiles = master_df['smiles'].drop_duplicates().to_frame()
unique_smiles['fp'] = unique_smiles['smiles'].apply(smiles_to_fingerprint)

# Convert list columns to a separate DataFrame of features
fp_features = pd.DataFrame(unique_smiles['fp'].tolist(), index=unique_smiles['smiles'])
fp_features.to_csv('../BioXAI_Hackathon/code_generated_outputs/fp_features_from_smiles.csv', index = False)

KeyError: 'smiles'

In [2]:
master_df

,sequenced_id,job_id,container_id,column_id,row_id,is_edge,compound,user_compound_id,compound_concentration,compound_concentration_unit,...,total_sequenced_reads,total_umi_count,ngenes3,n_mapped,percent_mapped,percent_rrna_removed,percent_mitochondrial,unassigned_multimapping,unassigned_nofeatures,percent_duplicated
0,116471544,tvc-qnu-012,1839025,1,1,True,e24cec84-a4a0-42b4-8914-67d02876a890,DMSO,0.15,%,...,2301152,888404,10858,1770243,86.07,10.63,5.215195,349502,109315,0.649494
1,116471545,tvc-qnu-012,1839025,1,2,True,9815b4fc-f243-497b-b5a1-408d0ac26e60,1306737,10000.00,nM,...,3087310,1222144,11371,2442721,86.80,8.85,4.305712,464527,122958,0.635829
2,116471546,tvc-qnu-012,1839025,1,3,True,1c40fdc5-e3ce-44b5-8a7d-8b4b004e9f3f,8632751,10000.00,nM,...,2497060,979693,11082,1930247,86.25,10.37,5.450177,380980,115455,0.653349
3,116471547,tvc-qnu-012,1839025,1,4,True,6c31658c-525c-4022-b807-470bc01963a0,8633329,10000.00,nM,...,2937675,1157313,11230,2298439,86.65,9.70,4.580870,449272,127200,0.646110
4,116471548,tvc-qnu-012,1839025,1,5,True,fb836044-82c5-4326-8afa-0ddbbe57738e,8633680,10000.00,nM,...,2735923,1059569,11476,2097834,85.77,10.60,5.702696,434923,142355,0.660163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32379,114756948,tvc-kdl-010,1809451,21,1,True,b29f5fe6-9b01-4568-a9af-51e47e8d0d2d,9244713,10000.00,nM,...,6207052,2232650,13010,4740284,85.37,10.54,4.783643,950935,346997,36.930000
32380,114756967,tvc-kdl-010,1809451,23,15,False,752496dc-1c58-4d49-93fd-3dff139d59ca,9264390,10000.00,nM,...,4888017,1752288,12358,3726539,85.47,10.80,5.102186,715358,250031,37.810000
32381,114756968,tvc-kdl-010,1809451,24,1,True,e7c56b40-f021-42e8-99e9-f7ae5e77ce29,9251671,10000.00,nM,...,6183224,2197323,12863,4700259,85.62,11.22,4.765526,914430,328096,37.800000
32382,114756974,tvc-kdl-010,1809451,24,13,True,b4288621-a655-4068-8497-7a1a401c9a46,Brefeldin-A,10000.00,nM,...,3876947,1368552,11109,2904125,85.38,12.27,5.003902,495876,169630,38.150000


In [ ]:
######## identify some genes which have differential expression across compounds
######## so that ML model can train from it 

# 1. Calculate the variance for each gene (columns)
gene_variances = counts_normalized.var(axis=0)

# 2. Sort the genes by variance in descending order
sorted_genes = gene_variances.sort_values(ascending=False)

# 3. View the top 10 most variable genes
print("Top 10 Highly Variable Genes in your dataset:")
print(sorted_genes.head(10))

# 4. Automatically pick the #1 top variable gene as your target 'y'
top_variable_gene = sorted_genes.index[0]
print(f"\nRecommended target gene (y): {top_variable_gene}")

# Extract it for your model
y = counts_normalized[top_variable_gene]

In [ ]:
############# Align fingerprint features with the target expression data (master dataframe) ######

# 1. Assuming fp_features has 'smiles' as its index
# Reset index to easily merge it back
fp_features_df = fp_features.reset_index().rename(columns={'index': 'smiles'})

# 2. Merge the fingerprint features back into your master data frame
final_ml_df = pd.merge(master_df, fp_features_df, on='smiles', how='inner')

# 3. Separate your chemical features (X) from your biological targets (y)
# Let's extract the fingerprint column names (usually integers 0 to 1023)
fp_cols = [col for col in fp_features.columns if col != 'smiles']

# Include physical properties alongside fingerprints for a stronger feature set
property_cols = ['molecular_weight', 'log_p', 'tpsa', 'num_rotatable_bonds', 'num_h_acceptors', 'num_h_donors']
feature_cols = property_cols + fp_cols

X = final_ml_df[feature_cols]

# For the target (y), choose a highly variable or key oncogene/marker from your normalized counts
# Example: let's pick a representative gene like 'TP53' or your top variable gene
target_gene = 'ENSG00000276070'  # Change this to an actual gene_id present in your columns
y = final_ml_df[target_gene]

In [ ]:
y

In [ ]:
##### Set a train-test split

from sklearn.model_selection import train_test_split

# A standard random split for a quick validation baseline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training shapes: {X_train.shape}, Test shapes: {X_test.shape}")

In [ ]:
#!pip install xgboost
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score

# Initialize an XGBoost Regressor to predict continuous normalized log-counts
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    random_state=42,
    n_jobs=-1
)

# Train the model
model.fit(X_train, y_train)

# Evaluate performance
preds = model.predict(X_test)
print(f"Test R² Score: {r2_score(y_test, preds):.3f}")
print(f"Test MSE: {mean_squared_error(y_test, preds):.3f}")

In [ ]:
##### SHAP explainability
# !pip install shap
import shap
import matplotlib.pyplot as plt

# 1. Initialize the Tree Explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

# 2. Plot the Summary Plot
# This displays the top features driving your gene predictions
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title(f"SHAP Feature Importance for Predicting {target_gene} Expression", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribution of unique genes detected
sns.histplot(data=df, x="ngenes3", kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Distribution of Unique Genes Detected")
axes[0].set_xlabel("Number of Genes (ngenes3)")

# 2. Distribution of total UMI counts (transcript abundance)
sns.histplot(data=df, x="total_umi_count", kde=True, ax=axes[1], color="salmon")
axes[1].set_title("Distribution of Total UMI Counts")
axes[1].set_xlabel("Total UMIs")

# 3. Distribution of Mitochondrial read percentage (Cell stress/viability)
sns.histplot(data=df, x="percent_mitochondrial", kde=True, ax=axes[2], color="lightgreen")
axes[2].set_title("Distribution of Mitochondrial Read %")
axes[2].set_xlabel("Mitochondrial %")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

# Boxplot comparing how different sample types alter the total number of genes expressed
sns.boxplot(
    data=df, 
    x="sample_type", 
    y="ngenes3", 
    hue="sample_type", 
    palette="Set2", 
    legend=False
)

plt.title("Impact of Sample Type on Total Expressed Genes", fontsize=14, pad=15)
plt.xlabel("Sample Type", fontsize=12)
plt.ylabel("Number of Genes Detected (ngenes3)", fontsize=12)
plt.xticks(rotation=15)

plt.show()

In [ ]:
# Scatter plot tracking complexity vs. sequencing depth, colored by control status
sns.scatterplot(
    data=df, 
    x="total_sequenced_reads", 
    y="ngenes3", 
    hue="is_neg_control", 
    alpha=0.8, 
    palette={True: "gray", False: "crimson"}
)

plt.title("Sequencing Depth vs. Number of Unique Genes Detected", fontsize=14)
plt.xlabel("Total Sequenced Reads", fontsize=12)
plt.ylabel("Unique Genes (ngenes3)", fontsize=12)
plt.legend(title="Is DMSO Control?")

plt.show()

In [ ]:
# Select key numeric QC metrics
qc_columns = [
    "total_sequenced_reads", "total_umi_count", "sequencing_saturation", 
    "ngenes3", "n_mapped", "percent_mapped", "percent_mitochondrial"
]

# Compute correlation matrix
corr_matrix = df[qc_columns].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix of Transcriptomic QC Metrics", fontsize=14, pad=15)

plt.show()

In [ ]:
compounds = df[~df['sample_type'].str.contains('Control', na=False)]['compound'].unique().tolist()

print(f"Found {len(compounds)} unique active compounds to look up.")

In [ ]:
compounds

In [ ]:
import time
def get_smiles_from_chembl(compound_name):
    """Queries ChEMBL API by molecule name and extracts canonical SMILES."""
    url = f"https://www.ebi.ac.uk/chembl/api/data/molecule.json"
    params = {
        'pref_name__iexact': compound_name,  # Case-insensitive name match
        'format': 'json'
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            # Check if any matching molecules were found
            if data.get('molecules'):
                # Extract canonical SMILES from the first record's structure sub-dictionary
                structures = data['molecules'][0].get('molecule_structures', {})
                if structures:
                    return structures.get('canonical_smiles', None)
        return None
    except Exception as e:
        return None

# Dictionary to hold our mapping results
smiles_mapping = {}

print("Starting ChEMBL structural lookup...")
for i, comp in enumerate(compounds):
    smiles = get_smiles_from_chembl(comp)
    smiles_mapping[comp] = smiles
    
    # Progress check
    if (i + 1) % 10 == 0 or (i + 1) == len(compounds):
        print(f"Processed {i + 1}/{len(compounds)} compounds...")
        
    # Politeness delay to prevent getting rate-limited by ChEMBL
    time.sleep(0.2)

print("Lookup complete!")

In [ ]:
get_smiles_from_chembl(comp)

In [ ]:
comp

In [ ]:
compounds

In [ ]:
###### look into gene_counts parquet
# !pip install scikit-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

# Load the count matrix
counts_df = pd.read_parquet("../gene_counts-2.parquet")

# If 'gene_symbol' or 'gene_id' is a column, set it as the index
if 'gene_id' in counts_df.columns:
    counts_df = counts_df.set_index('gene_id')
elif 'gene_symbol' in counts_df.columns:
    counts_df = counts_df.set_index('gene_symbol')

print(f"Matrix Shape: {counts_df.shape[0]} genes across {counts_df.shape[1]} samples.")

# Calculate sparsity (percentage of zeros in the dataset)
zero_percentage = (counts_df == 0).sum().sum() / counts_df.size * 100
print(f"Dataset Sparsity: {zero_percentage:.2f}% of the matrix is zeros.")

In [ ]:
counts_df

In [ ]:
import requests

def get_smiles_from_chembl_robust(compound_name):
    """Robust query targeting both primary preferred names and sub-nested synonyms."""
    base_url = "https://www.ebi.ac.uk/chembl/api/data/molecule.json"
    
    # Tier 1: Search by Primary Preferred Name
    try:
        response = requests.get(base_url, params={'pref_name__iexact': compound_name, 'format': 'json'}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('molecules'):
                structures = data['molecules'][0].get('molecule_structures', {})
                if structures and structures.get('canonical_smiles'):
                    return structures['canonical_smiles']
    except Exception:
        pass # Fall through to Tier 2 if network hiccups occur
        
    # Tier 2: Deep search through nested synonyms array if Tier 1 is blank
    try:
        response = requests.get(base_url, params={'molecule_synonyms__molecule_synonym__iexact': compound_name, 'format': 'json'}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('molecules'):
                structures = data['molecules'][0].get('molecule_structures', {})
                if structures and structures.get('canonical_smiles'):
                    return structures['canonical_smiles']
    except Exception:
        return None
        
    return None

smiles_mapping = {}

print("Starting ChEMBL structural lookup...")
for i, comp in enumerate(compounds):
    smiles = get_smiles_from_chembl(comp)
    smiles_mapping[comp] = smiles
    
    # Progress check
    if (i + 1) % 10 == 0 or (i + 1) == len(compounds):
        print(f"Processed {i + 1}/{len(compounds)} compounds...")
        
    # Politeness delay to prevent getting rate-limited by ChEMBL
    time.sleep(0.2)

print("Lookup complete!")

In [ ]:
smiles_mapping

smiles_lookup_df = pd.DataFrame.from_dict(smiles_mapping, orient='index', columns=['smiles'])

print(smiles_lookup_df)

smiles_lookup_df.to_csv('../BioXAI_Hackathon/all_mapping_smiles_for_active_compunds.csv', index = True)

In [ ]:
smiles_mapping

In [ ]:
###### make morgan footprints for smiles ####
#### 1. traditional approach: calculate morgan footprint
#### 2. ChemBerta Model: obtain embeddings
compunds_smiles = pd.read_csv('../BioXAI_Hackathon/all_mapping_smiles_for_active_compunds.csv')
compunds_smiles[compunds_smiles['smiles'].notna()]

In [ ]:
df.columns